In [1]:
import sys
sys.path.append("../")

from src.loaders.document_loader import EnterpriseDocumentLoader
from src.splitters.chunker import EnterpriseChunker

from langchain_ollama import OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

from pathlib import Path
import numpy as np

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = EnterpriseDocumentLoader()

documents = loader.load_directory("../datasets/military")

chunker = EnterpriseChunker(
    chunk_size=500,
    overlap=100,
    strategy="recursive"
)

chunks = chunker.split_documents(documents)

print("Chunks :", len(chunks))

Chunks : 29


In [3]:
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [4]:
vector = embedding_model.embed_query(
    "Radar calibration procedure"
)

print(type(vector))
print(len(vector))

<class 'list'>
768


In [5]:
embedding = embedding_model.embed_query(
    chunks[0].page_content
)

print(len(embedding))

768


In [6]:
texts = [
    chunk.page_content
    for chunk in chunks
]

embeddings = embedding_model.embed_documents(texts)

print(len(embeddings))
print(len(embeddings[0]))

29
768


In [7]:
chunk_vectors = []

for chunk, vector in zip(chunks, embeddings):

    chunk_vectors.append({

        "text": chunk.page_content,

        "metadata": chunk.metadata,

        "embedding": vector

    })

print(len(chunk_vectors))

29


In [8]:
from src.embeddings.embedding_manager import EmbeddingManager

embedder = EmbeddingManager(

    provider="ollama",

    model_name="nomic-embed-text"

)

vector = embedder.embed_query(

    "Military surveillance"

)

print(len(vector))

768


In [9]:
ollama_embed = EmbeddingManager(

    provider="ollama",

    model_name="nomic-embed-text"

)

In [10]:
hf_embed = EmbeddingManager(

    provider="huggingface",

    model_name="BAAI/bge-large-en-v1.5"

)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 16345.62it/s]


In [11]:
query = "Drone Maintenance"

vector = embedder.embed_query(query)

print(vector[:10])

[-0.015755508, -0.050708324, -0.18265991, -0.024411641, 0.06382694, -0.042650808, 0.0059205983, -0.02103948, 0.04049369, -0.027575681]


In [12]:
import pickle

with open("../vector_store/embeddings.pkl", "wb") as f:

    pickle.dump(chunk_vectors, f)

print("Embeddings Saved")

Embeddings Saved


In [15]:
query = "Drone Maintenance"

vector = ollama_embed.embed_query(query)

print(vector[:10])

vector = hf_embed.embed_query(query)

print(vector[:10])


[-0.015755508, -0.050708324, -0.18265991, -0.024411641, 0.06382694, -0.042650808, 0.0059205983, -0.02103948, 0.04049369, -0.027575681]
[0.004540547262877226, 0.013512182980775833, -0.023407932370901108, -0.0019648198504000902, -0.0006720814271830022, 0.0036542732268571854, -0.03444965183734894, 0.00624847412109375, 0.07543148845434189, -0.01718604564666748]
